# Lab 7 · IA sobre grafos para detectar anomalías de red
### Tema 7 · MU Gestión de Sistemas Complejos · Caso transversal: operador de telecomunicaciones

**Entorno:** SageMaker Notebook (kernel Python 3) · **Librerías:** pandas, NetworkX, scikit-learn, matplotlib
**Duración estimada:** 180 min · **Nivel:** intermedio-avanzado

---

## 1. Idea general del laboratorio

Cierra el bloque de redes (Labs 5 y 6). Ahora combinamos la **estructura** del grafo con datos de
**comportamiento** (tráfico, errores, latencia, reenrutamientos) para detectar nodos **anómalos** mediante
IA. Detectar anomalías es más sutil que un umbral fijo: un núcleo con mucho tráfico no es anómalo, es su
función; un acceso periférico con ese mismo tráfico sí lo sería. **La anomalía es relativa al contexto.**
Por eso usaremos un detector **global** (Isolation Forest) y z-scores **contextuales** por tipo de nodo.

### Objetivos didácticos
- Construir features que combinen estructura del grafo y comportamiento.
- Aplicar detección no supervisada (Isolation Forest).
- Calcular anomalías contextuales con z-scores por tipo de nodo.
- Validar con precisión y recall frente a una verdad conocida.
- Distinguir anomalía estadística de incidente operativo real.

> **Nota para el vídeo.** Cada celda está comentada paso a paso. `is_anomaly` contiene la "verdad"
> inyectada (4 nodos); en un caso real no la tendríamos: solo la usamos al final para evaluar.

In [ ]:
# ===========================================================================
# CELDA 1 · Dependencias
# ===========================================================================
import sys
!{sys.executable} -m pip install -q networkx scikit-learn matplotlib pandas
print("Dependencias listas")

## Parte A · Cargar y explorar las features

La tabla de medias por tipo de nodo es la **referencia** frente a la que algo será "anómalo en su contexto".

In [ ]:
# ===========================================================================
# CELDA 2 · Cargar node_features y ver medias por tipo de nodo
# ===========================================================================
import pandas as pd, numpy as np

RUTA = '.'   # local; o 's3://<tu-bucket>/graph'
feat = pd.read_csv(f'{RUTA}/node_features.csv')   # 1 fila por nodo
print("Dimensiones:", feat.shape)

# groupby('node_type')[...].mean(): media de cada variable de comportamiento
# por tipo de nodo. Esta tabla define que es "normal" en cada rol.
print(feat.groupby("node_type")[["utilization", "error_count_24h",
      "avg_latency_ms", "traffic_gbps"]].mean().round(2))

In [ ]:
# ===========================================================================
# CELDA 3 · Vistazo a las primeras filas (estructura + comportamiento)
# ===========================================================================
# Columnas estructurales: degree, betweenness, closeness (vienen del grafo).
# Columnas de comportamiento: traffic_gbps, utilization, error_count_24h,
# avg_latency_ms, reroute_events_24h. Y la etiqueta is_anomaly.
feat.head()

**Pregunta (vídeo).** Fíjate: el núcleo mueve mucho tráfico con baja latencia; el acceso mueve poco
tráfico pero acumula más errores y latencia. "Lo normal" depende del tipo de nodo.

## Parte B · Detección global con Isolation Forest

**Isolation Forest** aísla observaciones con árboles aleatorios: las anomalías, raras y distintas, se
aíslan con menos cortes. No supervisado. Lo aplicamos sobre el comportamiento estandarizado.

In [ ]:
# ===========================================================================
# CELDA 4 · Isolation Forest sobre las variables de comportamiento
# ===========================================================================
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Variables de comportamiento que alimentan el detector global.
vars_comportamiento = ['traffic_gbps', 'utilization',
                       'error_count_24h', 'avg_latency_ms', 'reroute_events_24h']

# StandardScaler estandariza cada variable (media 0, desviacion 1) para que
# todas pesen igual; si no, 'traffic_gbps' (decenas) dominaria a 'utilization' (0-1).
X = StandardScaler().fit_transform(feat[vars_comportamiento])

# contamination=0.1: proporcion ESPERADA de anomalias (~10%). random_state=42
# fija la aleatoriedad para resultados reproducibles.
iso = IsolationForest(contamination=0.1, random_state=42)

# fit_predict entrena y predice a la vez: devuelve -1 (anomalia) o 1 (normal).
feat['anomaly_score'] = iso.fit_predict(X)
# Convertimos a una columna binaria 1 = anomalia (mas comoda para contar/medir).
feat['anomaly_iso'] = (feat['anomaly_score'] == -1).astype(int)

print("Nodos marcados por Isolation Forest:")
print(feat[feat['anomaly_iso'] == 1]
      [['node_id', 'node_type', 'utilization', 'error_count_24h',
        'avg_latency_ms']].to_string(index=False))

> **Buena práctica.** `contamination` no se conoce exactamente en la realidad: conviene probar varios
> valores y revisar a mano los casos marcados, no fiarse del número.

## Parte C · Detección contextual con z-scores por tipo

Para cada variable, cuántas **desviaciones típicas** se aleja un nodo de la media **de su propio tipo**.

In [ ]:
# ===========================================================================
# CELDA 5 · z-score por tipo de nodo (anomalia CONTEXTUAL)
# ===========================================================================
def zscore_por_tipo(df, col):
    # media y desviacion de la columna DENTRO de cada tipo de nodo.
    # transform('mean') devuelve, para cada fila, la media de SU grupo.
    media = df.groupby('node_type')[col].transform('mean')
    std = df.groupby('node_type')[col].transform('std').replace(0, 1)  # evita /0
    # z = (valor - media_del_tipo) / desviacion_del_tipo
    return (df[col] - media) / std

# Calculamos el z-score contextual de tres variables clave.
for c in ['utilization', 'error_count_24h', 'avg_latency_ms']:
    feat[f'z_{c}'] = zscore_por_tipo(feat, c)

# z_max = el mayor de los tres z-scores de cada nodo (su "rareza" maxima).
feat['z_max'] = feat[['z_utilization', 'z_error_count_24h',
                      'z_avg_latency_ms']].max(axis=1)
# Marcamos anomalia contextual si z_max supera 2.5 (umbral habitual ~2-3 sigmas).
feat['anomaly_z'] = (feat['z_max'] > 2.5).astype(int)

print("Nodos marcados por z-score contextual:")
print(feat[feat['anomaly_z'] == 1][['node_id', 'node_type', 'z_max']]
      .sort_values('z_max', ascending=False).to_string(index=False))

## Parte D · Validar contra la verdad conocida

Usamos `is_anomaly` (que en realidad no tendríamos) para evaluar **precisión** y **recall** de cada detector.

In [ ]:
# ===========================================================================
# CELDA 6 · Precision, recall y matriz de confusion por metodo
# ===========================================================================
from sklearn.metrics import precision_score, recall_score, confusion_matrix

print("Anomalias reales inyectadas:")
print(feat[feat['is_anomaly'] == 1][['node_id', 'node_type']].to_string(index=False))
print()
# Para cada detector medimos:
#  - precision: de lo que marco, cuanto era realmente anomalo (pocos falsos +).
#  - recall: de lo realmente anomalo, cuanto detecto (pocos falsos -).
for metodo in ['anomaly_iso', 'anomaly_z']:
    p = precision_score(feat['is_anomaly'], feat[metodo], zero_division=0)
    r = recall_score(feat['is_anomaly'], feat[metodo], zero_division=0)
    print(f'{metodo}: precision {p:.2f}, recall {r:.2f}')
    # confusion_matrix: [[VN, FP], [FN, VP]]
    print(confusion_matrix(feat['is_anomaly'], feat[metodo]))
    print()

In [ ]:
# ===========================================================================
# CELDA 7 · Combinar detectores: AND (conservador) y OR (sensible)
# ===========================================================================
# AND: anomalia solo si la marcan AMBOS -> sube precision, baja recall.
feat['anomaly_combo'] = ((feat['anomaly_iso'] == 1) &
                         (feat['anomaly_z'] == 1)).astype(int)
p = precision_score(feat['is_anomaly'], feat['anomaly_combo'], zero_division=0)
r = recall_score(feat['is_anomaly'], feat['anomaly_combo'], zero_division=0)
print(f'combo (ambos / AND): precision {p:.2f}, recall {r:.2f}')

# OR: anomalia si la marca CUALQUIERA -> sube recall, baja precision.
feat['anomaly_or'] = ((feat['anomaly_iso'] == 1) |
                      (feat['anomaly_z'] == 1)).astype(int)
p2 = precision_score(feat['is_anomaly'], feat['anomaly_or'], zero_division=0)
r2 = recall_score(feat['is_anomaly'], feat['anomaly_or'], zero_division=0)
print(f'or (cualquiera / OR): precision {p2:.2f}, recall {r2:.2f}')

> **Buena práctica.** AND = menos falsas alarmas; OR = no se escapa ninguna. La elección depende de si es
> más caro investigar una falsa alarma o pasar por alto una anomalía real.

## Parte E · Visualizar las anomalías sobre el grafo

In [ ]:
# ===========================================================================
# CELDA 8 · Dibujar la red resaltando en rojo los nodos anomalos
# ===========================================================================
import networkx as nx, matplotlib.pyplot as plt

nodes = pd.read_csv(f'{RUTA}/network_nodes.csv')
edges = pd.read_csv(f'{RUTA}/network_edges.csv')
G = nx.Graph()
for _, r in edges.iterrows():
    G.add_edge(r['source'], r['target'])

# Conjunto de nodos marcados (usamos el OR para no perder ninguno).
anom = set(feat[feat['anomaly_or'] == 1]['node_id'])
# Rojo si es anomalo, azul si no.
colors = ['#c0392b' if n in anom else '#2980b9' for n in G.nodes()]

plt.figure(figsize=(13, 9))
pos = nx.spring_layout(G, seed=42, k=0.5)      # misma semilla -> disposicion estable
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=260)
# Etiquetamos solo los nodos anomalos para identificarlos.
nx.draw_networkx_labels(G, pos, {n: n for n in anom}, font_size=8)
plt.title('Nodos detectados como anomalos (en rojo)')
plt.axis('off'); plt.tight_layout(); plt.show()

## Parte F · De la anomalía a la acción

Una anomalía estadística **no es un incidente**: es una señal que merece investigación. Flujo:
detección automática → **triaje humano** → diagnóstico → acción o descarte → reentrenamiento.

> **Atención.** Sin triaje humano, un detector genera **fatiga de alertas** (tantos falsos positivos que se
> ignoran todos). La IA propone; la persona dispone.

In [ ]:
# ===========================================================================
# CELDA 9 · Ficha de cada nodo anomalo para apoyar el triaje
# ===========================================================================
# Mostramos el contexto de cada nodo marcado: tipo, region, sus metricas y si
# era una anomalia real (is_anomaly). Esto es lo que revisaria un tecnico.
detectados = feat[feat['anomaly_or'] == 1][
    ['node_id', 'node_type', 'region', 'utilization', 'error_count_24h',
     'avg_latency_ms', 'reroute_events_24h', 'z_max', 'is_anomaly']]
print(detectados.to_string(index=False))

## Parte G · Entregable · Parte H · Cierre

Entregable (5–7 págs.): medias por tipo y qué es "normal" en cada rol; nodos por Isolation Forest, por
z-score y por combinación; precisión/recall con matrices de confusión; visualización; hipótesis por nodo
(incidente real vs falso positivo); flujo de alerta; y reflexión anomalía vs incidente. Al terminar, **Stop**
la instancia.

---

### Cierre didáctico
"Lo normal" no es un número fijo sino algo **relativo al contexto** de cada nodo. El valor de un detector no
está en marcar casos, sino en integrarse en un flujo donde **una persona decide**. Cierras el bloque de
redes: construir (Lab 5), estresar (Lab 6) y vigilar con IA (Lab 7).